# GRU vs LSTM Flash Crash Prediction

This notebook:
1. Loads the prepared dataset
2. Converts tabular data into time sequences
3. Splits data into train/test sets
4. Trains GRU and LSTM models
5. Evaluates crash prediction performance


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout

## Step 1: Load Dataset

In [8]:
data = pd.read_csv('flash_crash_ready_dataset.csv')
print(data.shape)
data.head()

(234702, 17)


,Date,ticker,Open,High,Low,Close,Volume,VWAP,return,volatility,momentum,volume_change,vwap_diff,high_low_spread,open_close_return,turnover_change,crash_label
0,2007-12-11,ADANIPORTS,-0.072516,-0.075689,-0.081371,-0.085051,-0.310769,-0.077456,-0.973551,1.376206,0.024652,-0.051911,-1.874208,0.412591,-1.202954,-0.052548,0
1,2007-12-12,ADANIPORTS,-0.091463,-0.084847,-0.091184,-0.089251,-0.319921,-0.086699,-0.405895,0.810433,-0.264672,-0.042897,-0.627410,0.469612,0.221583,-0.044444,0
2,2007-12-13,ADANIPORTS,-0.088369,-0.052412,-0.085591,-0.053190,0.003825,-0.061475,3.276124,1.156212,0.261524,0.200725,2.024327,2.788292,3.505404,0.218506,0
3,2007-12-14,ADANIPORTS,-0.049742,-0.056228,-0.057781,-0.060720,-0.274494,-0.057427,-0.657854,1.255025,0.035254,-0.086290,-0.758106,-0.013398,-0.999947,-0.085737,0
4,2007-12-17,ADANIPORTS,-0.049703,-0.045543,-0.089025,-0.086367,-0.227913,-0.064057,-2.217551,1.561236,-0.183762,-0.012963,-5.381261,4.143309,-3.335218,-0.014680,0


## Step 2: Define Features

In [9]:
features = [
'Open','High','Low','Close','Volume','VWAP',
'return','volatility','momentum','volume_change','vwap_diff',
'high_low_spread','open_close_return','turnover_change'
]

## Step 3: Convert Dataset into Sequences

In [10]:
import numpy as np

sequence_length = 20

X = []
y = []

for ticker in data["ticker"].unique():

    stock_df = data[data["ticker"] == ticker].reset_index(drop=True)

    values = stock_df[features].values
    labels = stock_df["crash_label"].values

    if len(values) <= sequence_length:
        continue

    seq = np.lib.stride_tricks.sliding_window_view(
        values, (sequence_length, values.shape[1])
    )

    seq = seq.reshape(-1, sequence_length, values.shape[1])

    X.append(seq)

    y.append(labels[sequence_length:])

X = np.concatenate(X)
y = np.concatenate(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (233771, 20, 14)
y shape: (233722,)


In [11]:
min_len = min(len(X), len(y))

X = X[:min_len]
y = y[:min_len]

print(X.shape, y.shape)

(233722, 20, 14) (233722,)


## Step 4: Train Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

print(X_train.shape, X_test.shape)

(186977, 20, 14) (46745, 20, 14)


## Step 5: GRU Model

In [13]:
gru_model = Sequential([
    GRU(64, return_sequences=True, input_shape=(sequence_length, len(features))),
    Dropout(0.2),
    GRU(32),
    Dense(1, activation='sigmoid')
])

gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

c:\Users\kavan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 20, 64)         │        15,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,801 (96.88 KB)

 Trainable params: 24,801 (96.88 KB)

 Non-trainable params: 0 (0.00 B)

## Step 6: Train GRU

In [ ]:
gru_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    class_weight={0:1, 1:20}
)

Epoch 1/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.9882 - loss: 0.0587 - val_accuracy: 0.9894 - val_loss: 0.0545
Epoch 2/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.9895 - loss: 0.0537 - val_accuracy: 0.9894 - val_loss: 0.0538
Epoch 3/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.9895 - loss: 0.0534 - val_accuracy: 0.9894 - val_loss: 0.0536
Epoch 4/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9895 - loss: 0.0529 - val_accuracy: 0.9893 - val_loss: 0.0536
Epoch 5/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9895 - loss: 0.0524 - val_accuracy: 0.9893 - val_loss: 0.0530
Epoch 6/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9895 - loss: 0.0516 - val_accuracy: 0.9894 - val_loss: 0.0532
Epoch 7/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.9896 - loss: 0.0506 - val_accuracy: 0.9893 - val_loss: 0.0529
Epoch 8/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9896 - loss: 0

## Step 7: Evaluate GRU

In [15]:
pred = (gru_model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, pred))

1461/1461 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     46273
           1       0.32      0.04      0.07       472

    accuracy                           0.99     46745
   macro avg       0.66      0.52      0.53     46745
weighted avg       0.98      0.99      0.99     46745



## Step 8: LSTM Model

In [16]:
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(sequence_length, len(features))),
    Dropout(0.2),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

c:\Users\kavan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20, 64)         │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,673 (127.63 KB)

 Trainable params: 32,673 (127.63 KB)

 Non-trainable params: 0 (0.00 B)

## Step 9: Train LSTM

In [17]:
lstm_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.9885 - loss: 0.0580 - val_accuracy: 0.9893 - val_loss: 0.0542
Epoch 2/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.9895 - loss: 0.0539 - val_accuracy: 0.9893 - val_loss: 0.0556
Epoch 3/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - accuracy: 0.9895 - loss: 0.0534 - val_accuracy: 0.9893 - val_loss: 0.0538
Epoch 4/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 28s 12ms/step - accuracy: 0.9895 - loss: 0.0530 - val_accuracy: 0.9893 - val_loss: 0.0535
Epoch 5/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - accuracy: 0.9895 - loss: 0.0525 - val_accuracy: 0.9893 - val_loss: 0.0532
Epoch 6/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9895 - loss: 0.0522 - val_accuracy: 0.9894 - val_loss: 0.0535
Epoch 7/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9896 - loss: 0.0514 - val_accuracy: 0.9893 - val_loss: 0.0531
Epoch 8/10
2338/2338 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9896 - l

## Step 10: Evaluate LSTM

In [18]:
pred = (lstm_model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, pred))

1461/1461 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     46273
           1       0.39      0.02      0.04       472

    accuracy                           0.99     46745
   macro avg       0.69      0.51      0.52     46745
weighted avg       0.98      0.99      0.99     46745

